# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yashcodes07/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())



import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()
print(df.columns.tolist())
df.head(3)

Working dir: /content/flyrank-ml-internship
(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


In [2]:
# CODE — confirm the method choice makes sense given the data shape
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df))
print("\nFeature dtypes:")
print(df.dtypes)
print("\nClass balance (trend_direction):")
print(df["trend_direction"].value_counts(normalize=True).round(3))


Rows: 30000

Feature dtypes:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days   

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# CODE — build the split, adjusting to whichever columns actually exist
from sklearn.model_selection import GroupShuffleSplit

print("Available columns:", df.columns.tolist())

if "client_id" in df.columns:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
    train, test = df.iloc[train_idx], df.iloc[test_idx]
    print(f"Grouped split by client_id: {len(train)} train / {len(test)} test")
else:
    # fallback: simple holdout, sorted by any date-like column if present
    date_col = next((c for c in df.columns if "date" in c.lower()), None)
    if date_col:
        df_sorted = df.sort_values(date_col)
        cutoff = int(len(df_sorted) * 0.8)
        train, test = df_sorted.iloc[:cutoff], df_sorted.iloc[cutoff:]
        print(f"Time-aware split by {date_col}: {len(train)} train / {len(test)} test")
    else:
        from sklearn.model_selection import train_test_split
        train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["trend_direction"])
        print("No client_id or date column found — used stratified random split as fallback. "
              "Flag this in your text answer if it applies, since it's the weaker option.")


Available columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Grouped split by client_id: 23837 train / 6163 test


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training a Random Forest on the same features, same split, same metric (Precision@50) as the Week-4 baseline rule, so the comparison is apples-to-apples. [Fill in your baseline's exact feature list and rule logic here so the reader can see what's being beaten.]

In [4]:
# CODE — train RF, compute Precision@50 on the SAME split/metric as Week 4
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# TODO: swap in your actual Week-4 feature list
feature_cols = ["ctr", "content_age_days", "impressions_90d"]  # + any others you used
target_col = "trend_direction"

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

clf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

# Precision@50: treat "down" as the positive class of interest (declining pages to review)
probs = clf.predict_proba(X_test)
down_idx = list(clf.classes_).index("down")
test_scored = test.copy()
test_scored["down_prob"] = probs[:, down_idx]
top50 = test_scored.sort_values("down_prob", ascending=False).head(50)
precision_at_50 = (top50["trend_direction"] == "down").mean()

print(f"Baseline rule Precision@50 (Week 4): 0.240   <- fill in your real number")
print(f"Random Forest Precision@50 (this run): {precision_at_50:.3f}")
print(f"Lift: {precision_at_50 / 0.240:.2f}x")


Baseline rule Precision@50 (Week 4): 0.240   <- fill in your real number
Random Forest Precision@50 (this run): 0.460
Lift: 1.92x


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model's errors cluster around low-traffic pages, but the pattern is more subtle than it first appeared. Restricting to all 599 rows with near-zero traffic (impressions_90d ≤ 5, ctr = 0), the true labels are genuinely mixed — 37% flat, 26% new, 26% down, and smaller shares stable/up — and the model predicts new for only 31.2% of them, a mild over-prediction relative to the 26.4% that are truly new, not a dominant failure mode. The highest-confidence misses looked more dramatic than this because they're dominated by a small number of exact duplicate feature vectors — several pages share the identical (ctr=0.0, content_age_days, impressions_90d=3) combination despite having different true trend directions. With only three features, the model has no way to distinguish these pages from each other, so whichever single label it commits to for that combination is guaranteed wrong for the others sharing it. This points to a feature-richness limitation rather than a specific bias toward "new": low-traffic pages need additional signal (e.g. a longer historical window, or a trend feature computed over more than the current snapshot) to be separable at all. Combined with down's modest recall (0.33) and weak performance on flat/new/up (F1 0.18–0.27), the overall read stays the same — decent for coarse triage, not for fine-grained classification — but the specific mechanism behind the worst errors is duplicate feature collisions among low-traffic pages, not a systematic "predicts new for anything quiet" bias.

In [5]:
# CODE — confusion matrix + feature importance + worst misses
from sklearn.metrics import confusion_matrix, classification_report

preds = clf.predict(X_test)
print(classification_report(y_test, preds))

print("\nFeature importances:")
for feat, imp in sorted(zip(feature_cols, clf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")

# worst misses: high-confidence wrong predictions — most informative for error analysis
test_scored["predicted"] = preds
test_scored["pred_confidence"] = probs.max(axis=1)
misses = test_scored[test_scored["predicted"] != test_scored["trend_direction"]]
worst_misses = misses.sort_values("pred_confidence", ascending=False).head(10)
print("\nTop 10 most confident wrong predictions:")
print(worst_misses[["trend_direction", "predicted", "pred_confidence"] + feature_cols].to_string())

              precision    recall  f1-score   support

        down       0.62      0.33      0.43      3149
        flat       0.18      0.61      0.27       382
         new       0.20      0.17      0.18       302
      stable       0.35      0.47      0.40      1301
          up       0.17      0.20      0.18      1029

    accuracy                           0.35      6163
   macro avg       0.30      0.35      0.29      6163
weighted avg       0.44      0.35      0.36      6163


Feature importances:
  impressions_90d: 0.532
  content_age_days: 0.290
  ctr: 0.177

Top 10 most confident wrong predictions:
      trend_direction predicted  pred_confidence  ctr  content_age_days  impressions_90d
22907            down       new         0.855681  0.0               138                3
24769            flat       new         0.855681  0.0               138                3
2898             flat       new         0.855681  0.0               138                3
9247             down      

In [6]:
near_zero = test_scored[(test_scored["impressions_90d"] <= 5) & (test_scored["ctr"] == 0.0)]
print(f"Rows with near-zero traffic: {len(near_zero)}")
print(near_zero["trend_direction"].value_counts())
print(f"\nOf these, predicted as 'new':", (near_zero['predicted'] == 'new').mean().round(3))

Rows with near-zero traffic: 599
trend_direction
flat      222
new       158
down      157
stable     36
up         26
Name: count, dtype: int64

Of these, predicted as 'new': 0.312


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.